### **Scenario:**
You're a data engineer intern at a growing e-commerce startup. The analytics team handed you a CSV file with **8 million transaction records** from the past year.  

Unfortunately... it’s a mess:
- The data is inconsistent
- Column types are incorrect
- There are missing values, duplicates, mixed date formats
- And it's eating **way too much memory**

Your mission is to clean it up while keeping everything **memory-efficient and performant**.

---

### **Dataset Preview:**
The file is called `transactions.csv` (4 million rows) and contains:

| Column             | Description                                         |
|--------------------|-----------------------------------------------------|
| `customer_id`      | ID of the customer (some are missing)              |
| `transaction_id`   | Unique transaction string like `TXN1234567`        |
| `purchase_amount`  | Sometimes a float, sometimes a string, sometimes blank |
| `currency`         | Should be all 'USD', but has lowercase/missing     |
| `purchase_date`    | Mixed formats like `'2023/01/01'`, `'01-02-2023'`  |
| `product_id`       | Product ID, might be null                          |
| `product_category` | Category like 'Electronics', 'Books', messy casing |
| `is_returned`      | Values like `'yes'`, `True`, `'no'`, `False`, NaN  |

---

In [1]:
import pandas as pd
import psutil
import os

<hr>

**STEP 1:** To understand the memory impact, I’ll load the full dataset of 8 million rows and observe how much memory is used on a machine with 12GB of RAM. 

In [ ]:
# Memory usage before reading the file
process = psutil.Process(os.getpid())
print(f"The memory usage before reading the file is: {process.memory_info().rss / (1024**2)} MB")

# read the file
df = pd.read_csv('data/raw/transactions.csv')

# Memory usage after reading the file
process = psutil.Process(os.getpid())
print(f"The memory usage after reading the file is: {process.memory_info().rss / (1024**2)} MB")

118.25 MB
1081.50390625 MB


In [6]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8000000 entries, 0 to 7999999
Data columns (total 8 columns):
 #   Column            Dtype  
---  ------            -----  
 0   customer_id       float64
 1   transaction_id    object 
 2   purchase_amount   float64
 3   currency          object 
 4   purchase_date     object 
 5   product_id        object 
 6   product_category  object 
 7   is_returned       object 
dtypes: float64(2), object(6)
memory usage: 2.4 GB


**Memory Impact of Loading Full Dataset**

The memory usage before loading the file was **118.25 MB**, and after loading the full 8 million rows, it climbed up to **2.4 GB** — consuming over **1 GB of RAM**.

This shows that reading the entire dataset into memory on a machine with just 4GB of RAM would likely result in severe performance issues — including system freezing, crashing, or shutting down entirely. Obviously, this is not ideal for production environments or personal machines with limited memory resources.

This highlights the need for more **memory-efficient** data loading techniques, such as reading in chunks.




<hr>

**STEP 2: Restart the Kernel & Load a 100,000-Row Sample**

In this step, the kernel is restarted to ensure a fresh memory state. Then, a sample of the first 100,000 rows from the dataset is loaded.

Memory usage is recorded before and after loading the sample to assess the impact.

This subset will be used to explore, clean, and transform the data — allowing us to understand data quality, formats, and structure before scaling the logic to the full dataset.

In [2]:
# Memory usage before first 100k rows
process = psutil.Process(os.getpid())
print(f"The memory usage before reading the file is: {process.memory_info().rss / (1024**2)} MB")

# read the file
df = pd.read_csv('data/raw/transactions.csv', nrows=100000)

# Memory usage after first 100k rows
process = psutil.Process(os.getpid())
print(f"The memory usage of the first 100k rows is: {process.memory_info().rss / (1024**2)} MB")



The memory usage before reading the file is: 117.0703125 MB
The memory usage of the first 100k rows is: 130.37890625 MB


In [3]:
df.info(memory_usage='deep')

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 8 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   customer_id       100000 non-null  int64  
 1   transaction_id    100000 non-null  object 
 2   purchase_amount   71209 non-null   float64
 3   currency          66403 non-null   object 
 4   purchase_date     80096 non-null   object 
 5   product_id        80176 non-null   object 
 6   product_category  100000 non-null  object 
 7   is_returned       80037 non-null   object 
dtypes: float64(1), int64(1), object(6)
memory usage: 31.2 MB


**Memory Usage Observation**

Before reading the file, memory usage was approximately **117.63 MB**, and after loading *100,000 rows*, it increased to **130.37 MB**. Interestingly, running the same operation on a *4GB RAM machine* previously consumed over **300 MB**, whereas on a *12GB RAM machine*, the memory footprint was more than halved.

This highlights how hardware resources can significantly impact memory efficiency, even when reading data in chunks. It also reinforces that chunked reading is both necessary and more effective on machines with higher memory capacity.


<hr>

**STEP 3: Explore, Understand & Transform the Data**

In this step, we will explore the dataset to understand its structure, identify inconsistencies, and assess data quality. Based on these insights, we’ll clean and transform the data — with a strong focus on optimizing memory usage through proper data type conversions and handling of missing or inconsistent values.

In [4]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
# Drop duplicates
df = df.drop_duplicates()

In [6]:
# View first 10 rows
df.head(10)

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794,TXN2867825,120.50,USD,2023-03-15,P_234,books,no
1,10859,TXN1419610,NaN,usd,2023/01/01,P_234,Books,no
2,86819,TXN5614226,-15.00,USD,2023/01/01,P_456,toys,True
3,64885,TXN5108603,-15.00,USD,03.04.2023,P_234,Books,True
4,16264,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True
5,92385,TXN3341057,120.50,USD,NaN,P_234,TOYS,True
6,47193,TXN2719583,NaN,usd,01-02-2023,P_456,Electronics,yes
7,97497,TXN2458591,45.99,USD,2023/01/01,NaN,books,False
8,54130,TXN8078673,NaN,NaN,2023-03-15,P_234,Books,False
9,70262,TXN1533224,60.00,USD,NaN,P_456,TOYS,yes


I see a lot of NaN values in the purcahse amount column, The datatype of the column is a float64 dtype. The are very low chances that some string values were parsed as NaN.But we will check to see if there arent any

I will read the csv again and this time only 100 rows, there are a few nans in the first 100 rows so we can use that as a sample.

In [11]:
df2 = pd.read_csv('data/raw/transactions.csv', nrows=100000, dtype={"purchase_amount": "object"})

# See what was really there
df2.head(10)

,customer_id,transaction_id,purchase_amount,currency,purchase_date,product_id,product_category,is_returned
0,25794,TXN2867825,120.5,USD,2023-03-15,P_234,books,no
1,10859,TXN1419610,NaN,usd,2023/01/01,P_234,Books,no
2,86819,TXN5614226,-15.0,USD,2023/01/01,P_456,toys,True
3,64885,TXN5108603,-15.0,USD,03.04.2023,P_234,Books,True
4,16264,TXN4744854,45.99,USD,03.04.2023,P_456,Electronics,True
5,92385,TXN3341057,120.5,USD,NaN,P_234,TOYS,True
6,47193,TXN2719583,NaN,usd,01-02-2023,P_456,Electronics,yes
7,97497,TXN2458591,45.99,USD,2023/01/01,NaN,books,False
8,54130,TXN8078673,NaN,NaN,2023-03-15,P_234,Books,False
9,70262,TXN1533224,60.00,USD,NaN,P_456,TOYS,yes


In [12]:
print("sum of nulls in df1 purchase amount column",df['purchase_amount'].isna().sum())
print("sum of nulls in df2 purchase amount column",df2['purchase_amount'].isna().sum())


sum of nulls in df1 purchase amount column 28791
sum of nulls in df2 purchase amount column 28791


In [ ]:
df2.info()

Still nulls. We can see row 1, which is a nan value is consitents across both checks. The sum of null values remains the same across all both data frames, df which parsed the column as float, and df that parsed it as an object.